# Name: What Makes a Google Play App Successful?

Goal: Explore how category, type (free or paid), price, size, and last
update date relate to app installs and ratings, to help developers
decide what kind of new app to build.

## Asking Questions
Q1: Which category has the highest average rating?

Q2: Do free apps have more installs than paid apps on average?

Q3: Do apps updated more recently have higher average ratings?

Q4: Which category has the highest percentage of positive reviews?

Q5: Is there a relationship between app size and number of installs?

## Data Wrangling
### Gather


First, we import the libraries we will use throughout the project: NumPy, Pandas, Matplotlib and Seaborn.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



Then, we load the apps and reviews CSV files using relative paths, so the notebook runs on any computer.

In [2]:
df_apps_raw = pd.read_csv('data/raw/googleplaystore.csv')
df_reviews_raw = pd.read_csv('data/raw/googleplaystore_user_reviews.csv')


### Field Description

**Apps table (googleplaystore.csv)**
- `App`: the app name
- `Category`: the main category of the app (e.g. ART_AND_DESIGN)
- `Rating`: the average user rating, from 1 to 5
- `Reviews`: how many reviews the app has (a number)
- `Size`: the app size (e.g. 19M means 19 MB)
- `Installs`: about how many times the app was downloaded (e.g. 10,000+ means more than 10,000)
- `Type`: Free or Paid
- `Price`: how much the app costs if it's paid
- `Content Rating`: the age group the app is for (e.g. Everyone, Teen)
- `Genres`: one or more genres, separated by `;`
- `Last Updated`: the date of the last update
- `Current Ver`: the current version of the app
- `Android Ver`: the minimum Android version needed

**Reviews table (googleplaystore_user_reviews.csv)**
- `App`: the app name (links to the apps table)
- `Translated_Review`: the user's review text, translated to English
- `Sentiment`: Positive, Negative or Neutral
- `Sentiment_Polarity`: how positive the review is, from -1 (negative) to 1 (positive)
- `Sentiment_Subjectivity`: how personal the review is, from 0 (fact) to 1 (opinion)

### Assess

In [3]:
df_apps_raw.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


We also look at the first 5 rows of the reviews table.

In [4]:
df_reviews_raw.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


We check how many rows and columns each table has.

In [5]:
print('Apps table shape:')
print(df_apps_raw.shape)

Apps table shape:
(10841, 13)


In [6]:
print('Reviews table shape:')
print(df_reviews_raw.shape)

Reviews table shape:
(64295, 5)


info() shows the type of each column and how many values are not empty.

In [7]:
df_apps_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


In [8]:
df_reviews_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 64295 entries, 0 to 64294
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   App                     64295 non-null  str    
 1   Translated_Review       37427 non-null  str    
 2   Sentiment               37432 non-null  str    
 3   Sentiment_Polarity      37432 non-null  float64
 4   Sentiment_Subjectivity  37432 non-null  float64
dtypes: float64(2), str(3)
memory usage: 2.5 MB


We count the missing values in each column.

In [9]:
df_apps_raw.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [10]:
df_reviews_raw.isnull().sum()

App                           0
Translated_Review         26868
Sentiment                 26863
Sentiment_Polarity        26863
Sentiment_Subjectivity    26863
dtype: int64

We count the fully duplicated rows, then the duplicated app names.

In [11]:
print(df_apps_raw.duplicated().sum())


483


In [12]:
print(df_apps_raw['App'].duplicated().sum())

1181


We count the fully duplicated rows in the reviews table.

In [13]:
print(df_reviews_raw.duplicated().sum())

33616


describe() gives statistics for the number columns only.

In [14]:
df_apps_raw.describe()

,Rating
count,9367.000000
mean,4.193338
std,0.537431
min,1.000000
25%,4.000000
50%,4.300000
75%,4.500000
max,19.000000


In [15]:
df_reviews_raw.describe()

,Sentiment_Polarity,Sentiment_Subjectivity
count,37432.000000,37432.000000
mean,0.182146,0.492704
std,0.351301,0.259949
min,-1.000000,0.000000
25%,0.000000,0.357143
50%,0.150000,0.514286
75%,0.400000,0.650000
max,1.000000,1.000000


The max Rating is 19, but ratings go from 1 to 5, so we look for ratings above 5.

In [16]:
df_apps_raw[df_apps_raw['Rating']>5]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


We count the values of Type and Sentiment to find strange values.

In [17]:
df_apps_raw['Type'].value_counts()

Type
Free    10039
Paid      800
0           1
Name: count, dtype: int64

In [18]:
df_reviews_raw['Sentiment'].value_counts()

Sentiment
Positive    23998
Negative     8271
Neutral      5163
Name: count, dtype: int64

### Data Issues
- One broken row (10472): its values moved to the wrong columns
- Missing values: Rating (1,474), Current Ver (8), Android Ver (3), Type (1), Content Rating (1)
- Duplicate rows (483) and duplicate app names (1,181)
- Number columns stored as text: Reviews, Size, Installs, Price
- Date stored as text: Last Updated
- Columns we don't need: Android Ver, Current Ver
- Column names have spaces
- In reviews: 26,868 rows without text and 33,616 duplicate rows

### Clean

We make working copies so the original data stays safe.

In [19]:
df_apps = df_apps_raw.copy()
df_reviews = df_reviews_raw.copy()

We drop row 10472 because its Category value is missing, so all its values moved to the wrong place. We can't fix it safely, and it is only one row out of 10,841.

In [20]:
df_apps = df_apps.drop(index = 10472)
print(df_apps['Rating'].max())

5.0


We count the missing values again after dropping the broken row.

In [21]:
print(df_apps.isnull().sum())

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       0
Genres               0
Last Updated         0
Current Ver          8
Android Ver          2
dtype: int64


We keep the missing Rating values as NaN. Filling them would make fake ratings, and dropping them would lose other useful columns. Pandas ignores NaN when it calculates the mean.

One app has no Type. We look at this row to see if we can find its type from another column.

In [22]:
df_apps[df_apps['Type'].isnull()]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
9148,Command & Conquer: Rivals,FAMILY,NaN,0,Varies with device,0,NaN,0,Everyone 10+,Strategy,"June 28, 2018",Varies with device,Varies with device


We fill the missing Type with "Free", because this app's Price is 0.

In [23]:
df_apps['Type'] = df_apps['Type'].fillna('Free')
print(df_apps['Type'].value_counts(dropna=False))

Type
Free    10040
Paid      800
Name: count, dtype: int64


We drop 483 fully duplicated rows because they count the same app more than once.

In [24]:
print('Before:', df_apps.shape)
df_apps = df_apps.drop_duplicates()
print('After:', df_apps.shape)
print('Duplicated rows left:', df_apps.duplicated().sum())

Before: (10840, 13)


After: (10357, 13)
Duplicated rows left: 0


We convert Reviews from text to integer. It was text only because of the broken row we dropped.

In [25]:
df_apps['Reviews'] = df_apps['Reviews'].astype('int64')
print(df_apps['Reviews'].dtype)

int64


Some apps appear more than once with different values. We keep the row with the most Reviews, because it is the most recent one.

In [26]:
print('Duplicated app names before:', df_apps['App'].duplicated().sum())
df_apps = df_apps.sort_values('Reviews', ascending=False)
df_apps = df_apps.drop_duplicates(subset='App', keep='first')
print('Duplicated app names after:', df_apps['App'].duplicated().sum())
print('Shape:', df_apps.shape)

Duplicated app names before: 698
Duplicated app names after: 0
Shape: (9659, 13)


In the reviews table, we drop rows with no Sentiment, because they can't help answer Q4. We keep the 5 rows that have a Sentiment but no text, since Q4 only uses Sentiment.


In [27]:
print('Before:', df_reviews.shape)
df_reviews = df_reviews.dropna(subset=['Sentiment'])
print('After:', df_reviews.shape)
print(df_reviews.isnull().sum())

Before: (64295, 5)


After: (37432, 5)
App                       0
Translated_Review         5
Sentiment                 0
Sentiment_Polarity        0
Sentiment_Subjectivity    0
dtype: int64


We check the duplicated reviews and look at some of them before we decide.

In [28]:
print('Duplicated review rows:', df_reviews.duplicated().sum())
df_reviews[df_reviews.duplicated()].head(10)

Duplicated review rows: 7735


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
78,10 Best Foods for You,Good,Positive,0.70,0.600000
79,10 Best Foods for You,Good,Positive,0.70,0.600000
100,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
101,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
103,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
104,10 Best Foods for You,Best idea us,Positive,1.00,0.300000
105,10 Best Foods for You,Best way,Positive,1.00,0.300000
106,10 Best Foods for You,Amazing,Positive,0.60,0.900000
108,10 Best Foods for You,"Looking forward app,",Neutral,0.00,0.000000
109,10 Best Foods for You,It helpful site ! It help foods get !,Neutral,0.00,0.000000


Many reviews appear twice in the same order (for example, rows 100 to 109 repeat rows 0 to 9), so they were collected twice by mistake. We drop all duplicated reviews so they are not counted twice in Q4.

In [29]:
print('Before:', df_reviews.shape)
df_reviews = df_reviews.drop_duplicates()
print('After:', df_reviews.shape)

Before: (37432, 5)
After: (29697, 5)


We convert Last Updated from text to a real date, so we can sort it and get the year for Q3.

In [30]:
df_apps['Last Updated'] = pd.to_datetime(df_apps['Last Updated'], format='%B %d, %Y')
print(df_apps['Last Updated'].dtype)
print('Oldest update:', df_apps['Last Updated'].min())
print('Newest update:', df_apps['Last Updated'].max())


datetime64[us]
Oldest update: 2010-05-21 00:00:00
Newest update: 2018-08-08 00:00:00


We remove "," and "+" from Installs and convert it to integer. Note: each value is a minimum (10000 means "more than 10,000").

In [31]:
df_apps['Installs'] = df_apps['Installs'].str.replace(',', '', regex=False)
df_apps['Installs'] = df_apps['Installs'].str.replace('+', '', regex=False)
df_apps['Installs'] = df_apps['Installs'].astype('int64')
print(df_apps['Installs'].dtype)
print(df_apps['Installs'].value_counts().sort_index())

int64
Installs
0               15
1               67
5               82
10             385
50             204
100            709
500            328
1000           887
5000           468
10000         1030
50000          469
100000        1113
500000         504
1000000       1417
5000000        607
10000000       937
50000000       204
100000000      189
500000000       24
1000000000      20
Name: count, dtype: int64


We remove "$" from Price and convert it to a decimal number, because prices like 4.99 have cents.


In [32]:
df_apps['Price'] = df_apps['Price'].str.replace('$', '', regex=False).astype('float64')
print(df_apps['Price'].dtype)
print(df_apps['Price'].describe())

float64
count    9659.000000
mean        1.097231
std        16.851618
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       400.000000
Name: Price, dtype: float64


#### Outliers in Price
First, we look at the most expensive apps (price above $100).

In [33]:
df_apps[df_apps['Price'] > 100]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
5351,I am rich,LIFESTYLE,3.8,3547,1.8M,100000,Paid,399.99,Everyone,Lifestyle,2018-01-12,2.0,4.0.3 and up
5356,I Am Rich Premium,FINANCE,4.1,1867,4.7M,50000,Paid,399.99,Everyone,Finance,2017-11-12,1.6,4.0 and up
5354,I am Rich Plus,FAMILY,4.0,856,8.7M,10000,Paid,399.99,Everyone,Entertainment,2018-05-19,3.0,4.4 and up
4362,💎 I'm rich,LIFESTYLE,3.8,718,26M,10000,Paid,399.99,Everyone,Lifestyle,2018-03-11,1.0.0,4.4 and up
5359,I am rich(premium),FINANCE,3.5,472,965k,5000,Paid,399.99,Everyone,Finance,2017-05-01,3.4,4.4 and up
5355,I am rich VIP,LIFESTYLE,3.8,411,2.6M,10000,Paid,299.99,Everyone,Lifestyle,2018-07-21,1.1.1,4.3 and up
4367,I'm Rich - Trump Edition,LIFESTYLE,3.6,275,7.3M,10000,Paid,400.00,Everyone,Lifestyle,2018-05-03,1.0.1,4.1 and up
5366,I Am Rich,FAMILY,3.6,217,4.9M,10000,Paid,389.99,Everyone,Entertainment,2018-06-22,1.5,4.2 and up
5362,I Am Rich Pro,FAMILY,4.4,201,2.7M,5000,Paid,399.99,Everyone,Entertainment,2017-05-30,1.54,1.6 and up
5369,I am Rich,FINANCE,4.3,180,3.8M,5000,Paid,399.99,Everyone,Finance,2018-03-22,1.0,4.2 and up


Most of these apps are "I am rich" apps that cost about $400. They are real apps on Google Play, not data errors.
Next, we check the IQR method on Price.

In [34]:
q1 = df_apps['Price'].quantile(0.25)
q3 = df_apps['Price'].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr
print('Q1:', q1, '| Q3:', q3, '| IQR:', iqr, '| Upper limit:', upper_limit)
print('Apps above the limit:', (df_apps['Price'] > upper_limit).sum())

Q1: 0.0 | Q3: 0.0 | IQR: 0.0 | Upper limit: 0.0
Apps above the limit: 754


The upper limit is 0, because more than 75% of apps are free. To test it, we apply clip on a separate copy of Price, without changing df_apps.

In [35]:
price_clipped = df_apps['Price'].clip(upper=upper_limit)
print('Paid apps with price > 0 (original):', (df_apps['Price'] > 0).sum())
print('Paid apps with price > 0 (after clip):', (price_clipped > 0).sum())
print('Max price (original):', df_apps['Price'].max())
print('Max price (after clip):', price_clipped.max())

Paid apps with price > 0 (original): 754
Paid apps with price > 0 (after clip): 0


Max price (original): 400.0
Max price (after clip): 0.0


**Decision:** After clipping, every paid app has a price of 0, so clipping would destroy the Price column. We keep all prices as they are, and we will use the median when we compare prices.

#### Outliers in Reviews and Installs
Reviews and Installs also have very big values. We look at the apps with the most reviews to see if they are real.

In [36]:
df_apps.sort_values('Reviews', ascending=False)[['App', 'Category', 'Reviews', 'Installs']].head(5)

,App,Category,Reviews,Installs
2544,Facebook,SOCIAL,78158306,1000000000
336,WhatsApp Messenger,COMMUNICATION,69119316,1000000000
2604,Instagram,SOCIAL,66577446,1000000000
382,Messenger – Text and Video Chat for Free,COMMUNICATION,56646578,1000000000
1879,Clash of Clans,GAME,44893888,100000000


**Decision:** The biggest values belong to famous apps like Facebook and WhatsApp, so they are real and we keep them. Because these columns are very skewed, we will use the median (not only the mean) in the analysis.

#### Converting Size to MB
Size has values in MB ("19M"), in KB ("965k"), and "Varies with device". We write a function that converts one value to MB. "Varies with device" becomes NaN, because we don't know the real size.

In [37]:
# Convert one Size value to MB
def size_to_mb(size):
    if size == 'Varies with device':
        return np.nan                     # unknown size
    if 'M' in size:
        number = size.replace('M', '')    # '19M' -> '19'
        return float(number)              # '19' -> 19.0
    if 'k' in size:
        number = size.replace('k', '')    # '965k' -> '965'
        return float(number) / 1024       # KB -> MB

Now we apply the function to every value in Size, and check that the number of missing values equals the number of "Varies with device" values.

In [38]:
print('Varies with device:', (df_apps['Size'] == 'Varies with device').sum())
df_apps['Size'] = df_apps['Size'].apply(size_to_mb)
print('Missing Size after:', df_apps['Size'].isnull().sum())
print(df_apps['Size'].describe())

Varies with device: 1228
Missing Size after: 1228


count    8431.000000
mean       20.398075
std        21.828959
min         0.008301
25%         4.600000
50%        12.000000
75%        28.000000
max       100.000000
Name: Size, dtype: float64


#### Renaming and dropping columns
We rename the columns to lowercase names without spaces, so they are easier to write in code. We also rename Size to size_mb to show its unit.

In [39]:
df_apps = df_apps.rename(columns={
    'App': 'app',
    'Category': 'category',
    'Rating': 'rating',
    'Reviews': 'reviews',
    'Size': 'size_mb',
    'Installs': 'installs',
    'Type': 'type',
    'Price': 'price',
    'Content Rating': 'content_rating',
    'Genres': 'genres',
    'Last Updated': 'last_updated',
    'Current Ver': 'current_ver',
    'Android Ver': 'android_ver'
})

df_reviews = df_reviews.rename(columns={
    'App': 'app',
    'Translated_Review': 'review_text',
    'Sentiment': 'sentiment',
    'Sentiment_Polarity': 'polarity',
    'Sentiment_Subjectivity': 'subjectivity'
})

print(df_apps.columns.tolist())
print(df_reviews.columns.tolist())

['app', 'category', 'rating', 'reviews', 'size_mb', 'installs', 'type', 'price', 'content_rating', 'genres', 'last_updated', 'current_ver', 'android_ver']
['app', 'review_text', 'sentiment', 'polarity', 'subjectivity']


We drop current_ver and android_ver because none of our five questions uses them.

In [40]:
df_apps = df_apps.drop(columns=['current_ver', 'android_ver'])
print(df_apps.shape)

(9659, 11)


#### New columns
- update_year: the year of the last update, for Q3.
- main_genre: the first genre only, because some apps have two genres separated by ";".

In [41]:
df_apps['update_year'] = df_apps['last_updated'].dt.year
df_apps['main_genre'] = df_apps['genres'].apply(lambda g: g.split(';')[0])
df_apps[['app', 'last_updated', 'update_year', 'genres', 'main_genre']].head()

,app,last_updated,update_year,genres,main_genre
2544,Facebook,2018-08-03,2018,Social,Social
336,WhatsApp Messenger,2018-08-03,2018,Communication,Communication
2604,Instagram,2018-07-31,2018,Social,Social
382,Messenger – Text and Video Chat for Free,2018-08-01,2018,Communication,Communication
1879,Clash of Clans,2018-07-15,2018,Strategy,Strategy


#### Final check and saving the clean data
We check the final types and missing values, then save both clean tables in data/clean. We use index=False so the row numbers are not saved as an extra column.

In [42]:
df_apps.info()

<class 'pandas.DataFrame'>
Index: 9659 entries, 2544 to 7394
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   app             9659 non-null   str           
 1   category        9659 non-null   str           
 2   rating          8196 non-null   float64       
 3   reviews         9659 non-null   int64         
 4   size_mb         8431 non-null   float64       
 5   installs        9659 non-null   int64         
 6   type            9659 non-null   str           
 7   price           9659 non-null   float64       
 8   content_rating  9659 non-null   str           
 9   genres          9659 non-null   str           
 10  last_updated    9659 non-null   datetime64[us]
 11  update_year     9659 non-null   int32         
 12  main_genre      9659 non-null   str           
dtypes: datetime64[us](1), float64(3), int32(1), int64(2), str(6)
memory usage: 1018.7 KB


Now we save both clean tables.

In [43]:
df_apps.to_csv('data/clean/apps_clean.csv', index=False)
df_reviews.to_csv('data/clean/reviews_clean.csv', index=False)
print('Saved apps:', df_apps.shape)
print('Saved reviews:', df_reviews.shape)

Saved apps: (9659, 13)
Saved reviews: (29697, 5)


**Cleaning summary:** The apps table went from 10,841 rows to 9,659 unique apps, and the reviews table went from 64,295 rows to 29,697 real reviews. All number columns are numbers now, and the date is a real date. The only missing values left are in rating and size_mb, and we kept them on purpose.